# Module 2 — Evaluation & Analysis

Five mandatory evaluation protocols: topology map, circuit overlap, layer evolution, compositionality, and marginalization robustness.

In [ ]:
# Cell 1 – Setup
import subprocess, sys
for pkg in ["h5py", "umap-learn", "seaborn", "matplotlib", "numpy", "pandas", "tqdm"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)

ATLAS_HDF5  = "/content/drive/MyDrive/CSP-Atlas/dynamic_feature_atlas.h5"
STATS_JSON  = "/content/drive/MyDrive/CSP-Atlas/extraction_stats.json"

import sys
sys.path.insert(0, "/Users/piotrwilam/Code/CSP-Atlas/src")

import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns, h5py, json
from module2.io_utils import load_atlas_hdf5
from module2.metrics  import jaccard_similarity, entanglement_index

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

atlas = load_atlas_hdf5(ATLAS_HDF5)
pair_masks       = atlas["pair_masks"]
universal_masks  = atlas["universal_masks"]
metrics          = atlas["metrics"]
metadata         = atlas["metadata"]

print("Atlas loaded")
print(f"  Pairs          : {len(pair_masks)}")
print(f"  Universal AST  : {len(universal_masks['ast'])}")
print(f"  Universal Blt  : {len(universal_masks['builtin'])}")
print(f"  Metadata       : {metadata}")


## Protocol 1: Topology Map (UMAP)

In [ ]:
# Cell 2 – Protocol 1: Topology Map (UMAP)
# UMAP of Universal AST modules using Jaccard distance metric
import umap, numpy as np, matplotlib.pyplot as plt

REP_LAYER = 4

ast_names  = sorted(universal_masks["ast"].keys())
ast_vecs   = np.array([
    universal_masks["ast"][n][REP_LAYER].astype(np.float32)
    for n in ast_names
    if REP_LAYER in universal_masks["ast"][n]
], dtype=np.float32)

reducer = umap.UMAP(metric="jaccard", n_neighbors=10, min_dist=0.1,
                    random_state=42, verbose=False)
emb = reducer.fit_transform(ast_vecs)

fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(emb[:, 0], emb[:, 1], s=30, alpha=0.7)
for i, name in enumerate(ast_names[:len(emb)]):
    ax.annotate(name, (emb[i, 0], emb[i, 1]),
                fontsize=6, ha="center", va="bottom")
ax.set_title("UMAP — Universal AST Modules (Jaccard metric, layer 4)")
ax.set_xlabel("UMAP-1"); ax.set_ylabel("UMAP-2")
plt.tight_layout(); plt.show()
print(f"Plotted {len(emb)} AST modules")


In [ ]:
# Cell 3 – Protocol 1 continued: UMAP for Universal Builtin modules
import umap, numpy as np, matplotlib.pyplot as plt

blt_names = sorted(universal_masks["builtin"].keys())
blt_vecs  = np.array([
    universal_masks["builtin"][n][REP_LAYER].astype(np.float32)
    for n in blt_names
    if REP_LAYER in universal_masks["builtin"][n]
], dtype=np.float32)

reducer_b = umap.UMAP(metric="jaccard", n_neighbors=10, min_dist=0.1,
                       random_state=42, verbose=False)
emb_b = reducer_b.fit_transform(blt_vecs)

fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(emb_b[:, 0], emb_b[:, 1], s=30, alpha=0.7, color="orange")
for i, name in enumerate(blt_names[:len(emb_b)]):
    ax.annotate(name, (emb_b[i, 0], emb_b[i, 1]),
                fontsize=6, ha="center", va="bottom")
ax.set_title("UMAP — Universal Builtin Modules (Jaccard metric, layer 4)")
ax.set_xlabel("UMAP-1"); ax.set_ylabel("UMAP-2")
plt.tight_layout(); plt.show()


## Protocol 2: Circuit Overlap (Jaccard Heatmaps)

In [ ]:
# Cell 4 – Protocol 2: Circuit Overlap (Jaccard heatmaps)
import matplotlib.pyplot as plt, seaborn as sns, numpy as np

def plot_jaccard_heatmap(mat, names, title, max_labels=50):
    fig, ax = plt.subplots(figsize=(min(24, len(names)*0.35+2),
                                    min(24, len(names)*0.35+2)))
    show_labels = names if len(names) <= max_labels else False
    sns.heatmap(mat, ax=ax, vmin=0, vmax=1, cmap="viridis",
                xticklabels=show_labels, yticklabels=show_labels)
    ax.set_title(title)
    plt.tight_layout(); plt.show()

    off_diag = mat[np.triu_indices_from(mat, k=1)]
    print(f"{title}")
    print(f"  Mean Jaccard: {off_diag.mean():.4f}")
    print(f"  Max  Jaccard: {off_diag.max():.4f}")
    print(f"  Pairs with J>0.5: {(off_diag > 0.5).sum()}")

if "jaccard_ast_matrix" in metrics:
    ast_names_m = metrics.get("ast_names",
                  sorted(universal_masks["ast"].keys()))
    plot_jaccard_heatmap(metrics["jaccard_ast_matrix"], ast_names_m,
                         "Jaccard Similarity — Universal AST Modules (layer 4)")

if "jaccard_builtin_matrix" in metrics:
    blt_names_m = metrics.get("builtin_names",
                  sorted(universal_masks["builtin"].keys()))
    plot_jaccard_heatmap(metrics["jaccard_builtin_matrix"], blt_names_m,
                         "Jaccard Similarity — Universal Builtin Modules (layer 4)")


## Protocol 3: Layer Evolution

In [ ]:
# Cell 5 – Protocol 3: Layer Evolution
# How mean circuit size changes across layers for pair vs. universal modules
import numpy as np, matplotlib.pyplot as plt

layer_ids = sorted({lid for lm in pair_masks.values() for lid in lm})

pair_mean       = []
ast_univ_mean   = []
blt_univ_mean   = []

for lid in layer_ids:
    ps = [lm[lid].sum() for lm in pair_masks.values() if lid in lm]
    pair_mean.append(np.mean(ps) if ps else 0)

    as_ = [lm[lid].sum() for lm in universal_masks["ast"].values() if lid in lm]
    ast_univ_mean.append(np.mean(as_) if as_ else 0)

    bs = [lm[lid].sum() for lm in universal_masks["builtin"].values() if lid in lm]
    blt_univ_mean.append(np.mean(bs) if bs else 0)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(layer_ids, pair_mean,     marker="o", label="Pair Representations")
ax.plot(layer_ids, ast_univ_mean, marker="s", label="Universal AST")
ax.plot(layer_ids, blt_univ_mean, marker="^", label="Universal Builtin")
ax.set_xlabel("Layer"); ax.set_ylabel("Mean circuit size (neurons)")
ax.set_title("Circuit size evolution across layers")
ax.legend(); ax.grid(True)
plt.tight_layout(); plt.show()


## Protocol 4: Compositionality (Entanglement Index)

In [ ]:
# Cell 6 – Protocol 4: Compositionality (Entanglement Index)
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from module2.metrics import entanglement_index

REP_LAYER = 4
ei_rows = []

for (ast_n, blt_o), lm in pair_masks.items():
    if REP_LAYER not in lm:
        continue
    pm = lm[REP_LAYER]
    am = universal_masks["ast"].get(ast_n, {}).get(REP_LAYER)
    bm = universal_masks["builtin"].get(blt_o, {}).get(REP_LAYER)
    if am is None or bm is None:
        continue
    ei = entanglement_index(pm, am, bm)
    ei_rows.append({"ast_node": ast_n, "builtin_obj": blt_o,
                    "E_I": ei,
                    "pair_size": int(pm.sum()),
                    "ast_size":  int(am.sum()),
                    "blt_size":  int(bm.sum())})

ei_df = pd.DataFrame(ei_rows)
print(f"E_I computed for {len(ei_df)} pairs at layer {REP_LAYER}")
print(ei_df["E_I"].describe())

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(ei_df["E_I"].values, bins=40, edgecolor="black")
ax.set_xlabel("Entanglement Index (E_I)")
ax.set_ylabel("Count")
ax.set_title("Distribution of Entanglement Index across all pairs (layer 4)\n"
             "E_I=0 → perfectly compositional | E_I=1 → fully unique")
plt.tight_layout(); plt.show()

# Scatter: pair_size vs E_I
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(ei_df["pair_size"], ei_df["E_I"], alpha=0.4, s=20)
ax.set_xlabel("Pair circuit size"); ax.set_ylabel("E_I")
ax.set_title("Circuit size vs. Entanglement Index")
plt.tight_layout(); plt.show()


In [ ]:
# Cell 7 – Protocol 4 continued: most/least compositional pairs
print("Most compositional pairs (lowest E_I):")
print(ei_df.nsmallest(10, "E_I")[["ast_node","builtin_obj","E_I"]].to_string(index=False))

print("\nLeast compositional pairs (highest E_I):")
print(ei_df.nlargest(10, "E_I")[["ast_node","builtin_obj","E_I"]].to_string(index=False))


## Protocol 5: Marginalization Robustness

In [ ]:
# Cell 8 – Protocol 5: Marginalization Robustness
# For each Universal AST module: compute circuit size as builtins are progressively added
# Shows how quickly the universal module stabilises
import numpy as np, matplotlib.pyplot as plt, random

sample_ast_nodes = random.sample(sorted(universal_masks["ast"].keys()),
                                 min(5, len(universal_masks["ast"])))
REP_LAYER = 4

fig, ax = plt.subplots(figsize=(9, 5))

for ast_n in sample_ast_nodes:
    # Find all builtins available for this AST node
    relevant_blt = [b for (a, b) in pair_masks if a == ast_n
                    and REP_LAYER in pair_masks[(a, b)]]
    if len(relevant_blt) < 2:
        continue

    random.shuffle(relevant_blt)
    sizes = []
    running = None

    for blt in relevant_blt:
        pm = pair_masks[(ast_n, blt)][REP_LAYER]
        running = pm.copy() if running is None else np.logical_and(running, pm)
        sizes.append(int(running.sum()))

    ax.plot(range(1, len(sizes)+1), sizes, marker="o", label=ast_n, alpha=0.8)

ax.set_xlabel("Number of builtins intersected")
ax.set_ylabel("Universal AST circuit size (neurons)")
ax.set_title("Marginalization Robustness — Universal AST circuit convergence")
ax.legend(fontsize=8); ax.grid(True)
plt.tight_layout(); plt.show()


In [ ]:
# Cell 9 – Protocol 5 continued: Builtin marginalization robustness
import numpy as np, matplotlib.pyplot as plt, random

sample_blt_objs = random.sample(sorted(universal_masks["builtin"].keys()),
                                 min(5, len(universal_masks["builtin"])))

fig, ax = plt.subplots(figsize=(9, 5))

for blt_o in sample_blt_objs:
    relevant_ast = [a for (a, b) in pair_masks if b == blt_o
                    and REP_LAYER in pair_masks[(a, b)]]
    if len(relevant_ast) < 2:
        continue

    random.shuffle(relevant_ast)
    sizes = []
    running = None

    for ast_n in relevant_ast:
        pm = pair_masks[(ast_n, blt_o)][REP_LAYER]
        running = pm.copy() if running is None else np.logical_and(running, pm)
        sizes.append(int(running.sum()))

    ax.plot(range(1, len(sizes)+1), sizes, marker="s", label=blt_o, alpha=0.8)

ax.set_xlabel("Number of AST nodes intersected")
ax.set_ylabel("Universal Builtin circuit size (neurons)")
ax.set_title("Marginalization Robustness — Universal Builtin circuit convergence")
ax.legend(fontsize=8); ax.grid(True)
plt.tight_layout(); plt.show()


## Ockham Index (O_I)

In [ ]:
# Cell 10 – Ockham Index (Jaccard distance between Universal Modules)
from module2.metrics import jaccard_distance
import pandas as pd, numpy as np

REP_LAYER = 4

# AST x AST Ockham distances
ast_names_sorted = sorted(universal_masks["ast"].keys())
oi_rows = []
for i, a1 in enumerate(ast_names_sorted):
    m1 = universal_masks["ast"][a1].get(REP_LAYER)
    if m1 is None: continue
    for a2 in ast_names_sorted[i+1:]:
        m2 = universal_masks["ast"][a2].get(REP_LAYER)
        if m2 is None: continue
        oi_rows.append({"a": a1, "b": a2,
                        "O_I": jaccard_distance(m1, m2)})

oi_df = pd.DataFrame(oi_rows)
print("Ockham Index (AST x AST) — summary:")
print(oi_df["O_I"].describe())
print("\nMost similar AST node pairs (low O_I = high overlap):")
print(oi_df.nsmallest(10, "O_I")[["a","b","O_I"]].to_string(index=False))


## Summary Report

In [ ]:
# Cell 11 – Summary report
import json, datetime

report = {
    "generated_at"        : datetime.datetime.utcnow().isoformat() + "Z",
    "n_pairs"             : len(pair_masks),
    "n_universal_ast"     : len(universal_masks["ast"]),
    "n_universal_builtin" : len(universal_masks["builtin"]),
    "mean_EI"             : float(ei_df["E_I"].mean()) if len(ei_df) else None,
    "fraction_EI_lt_0.2"  : float((ei_df["E_I"] < 0.2).mean()) if len(ei_df) else None,
    "mean_OI_ast"         : float(oi_df["O_I"].mean()) if len(oi_df) else None,
    "metadata"            : {str(k): str(v) for k, v in metadata.items()},
}

print(json.dumps(report, indent=2))

# Close atlas HDF5 handle if open
if hasattr(atlas.get("handle"), "close"):
    atlas["handle"].close()
    print("HDF5 handle closed")


In [ ]:
# Cell 12 – Done
print("Module 2 evaluation complete.")
print("All 5 protocols executed:")
print("  1. Topology Map (UMAP)")
print("  2. Circuit Overlap (Jaccard heatmaps)")
print("  3. Layer Evolution")
print("  4. Compositionality (Entanglement Index)")
print("  5. Marginalization Robustness")
